# SPINE-GPE v7 — PNADc Historical Backcast Publication Rerun v1.2.0

Este notebook executa o hardening corrigido com perfil de publicação. O `RUN_ID` é persistido para permitir retomada do bootstrap após interrupções.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import datetime as dt
import json
import subprocess
import sys
import pandas as pd

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPTS = ROOT / 'scripts'
SCRIPT = SCRIPTS / 'SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.2.0.py'
REQ = SCRIPTS / 'requirements_SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.2.0.txt'

assert SCRIPT.exists(), SCRIPT
assert REQ.exists(), REQ
print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.2.0.py


## 1. Dependências

In [2]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)],
    text=True,
    capture_output=True,
    check=False,
)
print(install.stdout)
print(install.stderr)
print('Install exit code:', install.returncode)
assert install.returncode == 0



Install exit code: 0


## 2. Run ID persistente

O mesmo ID deve ser reutilizado com `--resume-bootstrap` caso a execução seja interrompida.

In [3]:
RUN_ID_FILE = ROOT / '00_admin' / 'PNADC_BACKCAST_PUBLICATION_RUN_ID_v120.txt'
RUN_ID_FILE.parent.mkdir(parents=True, exist_ok=True)

if RUN_ID_FILE.exists():
    RUN_ID = RUN_ID_FILE.read_text(encoding='utf-8').strip()
else:
    RUN_ID = 'publication_' + dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_v120'
    RUN_ID_FILE.write_text(RUN_ID + chr(10), encoding='utf-8')

print('RUN_ID:', RUN_ID)
print('Arquivo de retomada:', RUN_ID_FILE)

RUN_ID: publication_20260724T185511Z_v120
Arquivo de retomada: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_BACKCAST_PUBLICATION_RUN_ID_v120.txt


## 3. Auditoria

Inclua em `LAYOUT_ROOTS` diretórios adicionais que contenham os layouts oficiais anuais. A lista pode permanecer vazia; nesse caso, a limitação documental será preservada como warning.

In [4]:
LAYOUT_ROOTS = [
    # ROOT / '01_raw' / '10_ibge' / 'pnadc_layouts_anuais',
]

cmd_audit = [
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'audit',
    '--profile', 'publication',
    '--strict',
]
for layout_root in LAYOUT_ROOTS:
    cmd_audit.extend(['--layout-root', str(layout_root)])

audit = subprocess.run(cmd_audit, text=True, capture_output=True, check=False)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)
assert audit.returncode == 0

{
  "run_id": "20260724T185522Z",
  "script_version": "1.2.0",
  "schema_version": "spine-gpe-v7-pnadc-historical-backcast-hardening-1.2.0",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [
    {
      "test_id": "layout.independent_year_versions",
      "status": "WARN",
      "severity": "high",
      "message": "Somente uma ou nenhuma versão anual independente foi localizada; equivalência semântica anual permanece documentada como limitação.",
      "observed": [
        "2022"
      ],
      "expected": ">=2 annual versions",
      "evidence": null
    }
  ],
  "layout_matrix": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_layout_equivalence_matrix_20260724T185522Z.csv",
  "upstream_lock": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json",
  "upstream_lock_sha256": "d9b3e0638acdbe5a44adbfa5046ec6622b5352da8aa44a1

## 4. Rerun final — 500 réplicas

O checkpoint é atualizado a cada 25 réplicas. O perfil de publicação exige pelo menos 95% de réplicas bem-sucedidas.

In [5]:
cmd_full = [
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'full',
    '--profile', 'publication',
    '--run-id', RUN_ID,
    '--bootstrap-reps', '500',
    '--bootstrap-checkpoint-every', '25',
    '--bootstrap-min-success-rate', '0.95',
    '--bootstrap-seed', '20260724',
    '--calibration-folds', '5',
    '--bootstrap-calibration-folds', '3',
    '--mca-components', '8',
    '--cluster-k-min', '4',
    '--cluster-k-max', '8',
    '--knn-min-temporal-correlation', '0.90',
    '--knn-max-relative-difference', '0.10',
    '--strict',
]
for layout_root in LAYOUT_ROOTS:
    cmd_full.extend(['--layout-root', str(layout_root)])

full = subprocess.run(cmd_full, text=True, capture_output=True, check=False)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)
assert full.returncode == 0

{
  "run_id": "publication_20260724T185511Z_v120",
  "script_version": "1.2.0",
  "schema_version": "spine-gpe-v7-pnadc-historical-backcast-hardening-1.2.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-backcast-validation-1.2.0",
  "model_schema_version": "spine-gpe-v7-pnadc-backcast-model-1.2.0",
  "mode": "full",
  "status": "FINAL_CERTIFIED",
  "critical_failures": [],
  "warnings": [
    {
      "test_id": "layout.independent_year_versions",
      "status": "WARN",
      "severity": "high",
      "message": "Somente uma ou nenhuma versão anual independente foi localizada; equivalência semântica anual permanece documentada como limitação.",
      "observed": [
        "2022"
      ],
      "expected": ">=2 annual versions",
      "evidence": null
    }
  ],
  "upstream_historical_lock": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json",
  "upstream_historical_lock_sha256": "d9b3e0638acdbe5a44adbfa5046ec6622b5352da8

## 5. Retomada após interrupção

Execute esta célula somente quando a célula anterior tiver sido interrompida. Ela reutiliza o checkpoint do mesmo `RUN_ID`.

In [6]:
cmd_resume = cmd_full + ['--resume-bootstrap']
print('Comando de retomada preparado para:', RUN_ID)
# resume = subprocess.run(cmd_resume, text=True, capture_output=True, check=False)
# print(resume.stdout)
# print(resume.stderr)
# print('Resume exit code:', resume.returncode)
# assert resume.returncode == 0

Comando de retomada preparado para: publication_20260724T185511Z_v120


## 6. Inspeção dos artefatos

In [7]:
TABLE = ROOT / '05_outputs' / 'tables' / 'pnadc_historical_backcast_hardening'

paths = {
    'metrics': TABLE / f'pnadc_proxy_temporal_calibrated_metrics_{RUN_ID}.csv',
    'goldens': TABLE / f'pnadc_proxy_direct_aggregate_goldens_{RUN_ID}.csv',
    'knn_aggregate': TABLE / f'pnadc_logit_knn_aggregate_stability_{RUN_ID}.csv',
    'support_summary': TABLE / f'pnadc_historical_backcast_support_summary_{RUN_ID}.csv',
    'bootstrap_summary': TABLE / f'pnadc_proxy_model_uncertainty_{RUN_ID}.csv',
    'final_estimates': TABLE / f'pnadc_historical_backcast_final_estimates_{RUN_ID}.csv',
    'mca_cells': TABLE / f'pnadc_mca_historical_cell_coordinates_{RUN_ID}.csv',
    'support': TABLE / f'pnadc_historical_transport_support_{RUN_ID}.csv',
}
for name, path in paths.items():
    print(name, '=>', path, '| exists=', path.exists())
    assert path.exists(), path

metrics => /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_proxy_temporal_calibrated_metrics_publication_20260724T185511Z_v120.csv | exists= True
goldens => /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_proxy_direct_aggregate_goldens_publication_20260724T185511Z_v120.csv | exists= True
knn_aggregate => /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_logit_knn_aggregate_stability_publication_20260724T185511Z_v120.csv | exists= True
support_summary => /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_historical_backcast_support_summary_publication_20260724T185511Z_v120.csv | exists= True
bootstrap_summary => /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_proxy_mo

In [8]:
metrics = pd.read_csv(paths['metrics'])
goldens = pd.read_csv(paths['goldens'])
knn_aggregate = pd.read_csv(paths['knn_aggregate'])
bootstrap_summary = pd.read_csv(paths['bootstrap_summary'])
final_estimates = pd.read_csv(paths['final_estimates'])

print('MÉTRICAS TEMPORAIS')
display(metrics)
print('GOLDEN TOTALS')
display(goldens)
print('KNN AGREGADO')
display(knn_aggregate)
print('BOOTSTRAP')
display(bootstrap_summary)
print('ESTIMATIVAS FINAIS')
display(final_estimates)

MÉTRICAS TEMPORAIS


,model,train_year,test_year,n,n_positive_unweighted,weighted_prevalence,roc_auc,average_precision,brier,null_brier,...,log_loss,ece_10,ece_relative_prevalence,calibration_intercept,calibration_slope,calibration_status,calibration_method,calibration_n_clipped,probability_min,probability_max
0,mapping_2022,2022,2024,184157,778,0.005505,0.978834,0.363988,0.004009,0.005475,...,0.016646,0.000214,0.038874,-0.097745,0.938086,ESTIMATED,weighted_likelihood_lbfgsb,32687,0.0,0.939842
1,mapping_2024,2024,2022,178163,691,0.005208,0.979361,0.316580,0.003948,0.005181,...,0.016682,0.000139,0.026638,-0.141644,0.949756,ESTIMATED,weighted_likelihood_lbfgsb,109141,0.0,0.605469


GOLDEN TOTALS


,year,model,direct_total,direct_total_se,direct_share,direct_share_se,model_total,model_total_se_conditional,model_share,model_share_se_conditional,difference_absolute,difference_relative,predicted_to_observed_ratio
0,2022,mapping_2022,445866.911584,26056.331061,0.005208,0.000302,442641.594557,14564.168938,0.005171,0.000166,-3225.317027,-0.007234,0.992766
1,2022,mapping_2024,445866.911584,26056.331061,0.005208,0.000302,453191.798549,13603.319894,0.005294,0.000156,7324.886965,0.016428,1.016428
2,2022,mapping_pooled,445866.911584,26056.331061,0.005208,0.000302,451993.207293,13886.090428,0.005280,0.000158,6126.295709,0.013740,1.013740
3,2024,mapping_2022,487284.887116,25149.948865,0.005505,0.000283,470387.153259,13164.554609,0.005314,0.000146,-16897.733857,-0.034677,0.965323
4,2024,mapping_2024,487284.887116,25149.948865,0.005505,0.000283,488093.388075,13915.516473,0.005514,0.000154,808.500958,0.001659,1.001659
5,2024,mapping_pooled,487284.887116,25149.948865,0.005505,0.000283,482905.058534,13529.423674,0.005456,0.000150,-4379.828582,-0.008988,0.991012


KNN AGREGADO


,n_periods,temporal_pearson_total,temporal_spearman_total,mean_absolute_relative_total_difference,max_absolute_relative_total_difference,mean_micro_spearman,mean_weighted_pearson_probability,mean_top10_weighted_overlap
0,12,0.996589,1.0,0.016189,0.038285,0.230688,0.941517,0.653618


BOOTSTRAP


,source_period,model_bootstrap_reps,model_total_mean,model_total_sd,model_total_p025,model_total_median,model_total_p975,model_share_mean,model_share_sd,model_share_p025,model_share_p975,model_total_cv,model_share_cv
0,2019q1,500,315497.471170,12934.960851,288660.514129,314876.686480,340100.453307,0.003933,0.000161,0.003598,0.004240,0.040999,0.040999
1,2019q2,500,321629.998730,12911.477873,295077.199930,321504.504995,346500.997358,0.003952,0.000159,0.003626,0.004257,0.040144,0.040144
2,2019q3,500,340901.621322,13181.444200,314031.089865,340341.917461,367286.519012,0.004163,0.000161,0.003835,0.004486,0.038666,0.038666
3,2019q4,500,356543.511565,13722.051670,327564.491198,356464.246518,383177.841198,0.004313,0.000166,0.003962,0.004635,0.038486,0.038486
4,2020q1,500,353119.720890,13267.023809,326209.380788,352880.924339,379517.209842,0.004397,0.000165,0.004062,0.004725,0.037571,0.037571
5,2020q2,500,350872.098710,11680.212139,326879.676361,350616.671671,373682.621834,0.004952,0.000165,0.004613,0.005274,0.033289,0.033289
6,2020q3,500,357569.542730,11929.324610,333357.858019,357686.958787,381558.625824,0.005055,0.000169,0.004713,0.005394,0.033362,0.033362
7,2020q4,500,357501.712875,11959.353382,333207.948289,356934.493730,381405.340729,0.004819,0.000161,0.004491,0.005141,0.033453,0.033453
8,2021q1,500,382517.311492,12462.018194,358637.339989,382341.982278,406400.836465,0.005149,0.000168,0.004828,0.005471,0.032579,0.032579
9,2021q2,500,409246.360465,13116.393608,383979.252964,409351.256638,435642.560648,0.005350,0.000171,0.005020,0.005695,0.032050,0.032050


ESTIMATIVAS FINAIS


,source_period,total_2022,total_se_survey_2022,share_2022,share_se_survey_2022,total_2024,total_se_survey_2024,share_2024,share_se_survey_2024,total_pooled,...,model_share_cv,total_se_survey_plus_model,share_se_survey_plus_model,share_ci95_lower_survey_plus_model,share_ci95_upper_survey_plus_model,total_ci95_lower_survey_plus_model,total_ci95_upper_survey_plus_model,claim_status,direct_platform_observed,evidence_tier
0,2019q1,307848.950745,7673.296916,0.003837,0.000094,321739.166716,7494.880663,0.004011,0.000092,317478.603902,...,0.040999,14925.492344,0.000185,0.003595,0.004320,288224.638908,346732.568897,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
1,2019q2,310956.762109,6930.421512,0.003821,0.000084,329373.770203,7590.596554,0.004047,0.000092,322971.633478,...,0.040144,14806.707882,0.000181,0.003613,0.004324,293950.486030,351992.780926,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
2,2019q3,332537.699466,8062.022684,0.004061,0.000097,347906.341214,8472.661143,0.004249,0.000102,342236.920247,...,0.038666,15552.314177,0.000189,0.003809,0.004550,311754.384459,372719.456035,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
3,2019q4,348864.331314,9976.049668,0.004220,0.000117,361810.591683,9714.643998,0.004376,0.000114,358152.489032,...,0.038486,16836.283512,0.000201,0.003937,0.004727,325153.373349,391151.604714,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
4,2020q1,346095.794941,10344.648076,0.004309,0.000126,357607.258125,10177.590542,0.004453,0.000124,355182.969437,...,0.037571,16717.945836,0.000207,0.004017,0.004827,322415.795598,387950.143275,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
5,2020q2,343226.854878,13772.841006,0.004844,0.000192,355449.121986,13954.212102,0.005017,0.000195,352728.846651,...,0.033289,18100.568673,0.000254,0.004481,0.005475,317251.732051,388205.961251,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
6,2020q3,350976.691152,15152.451472,0.004962,0.000211,359452.277036,14689.958438,0.005082,0.000204,360248.481674,...,0.033362,19110.924417,0.000267,0.004569,0.005617,322791.069816,397705.893531,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
7,2020q4,349085.626281,14675.164721,0.004705,0.000196,360010.213908,13841.860917,0.004853,0.000184,358670.590713,...,0.033453,18592.882193,0.000249,0.004347,0.005322,322228.541614,395112.639811,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
8,2021q1,370011.575193,17555.627446,0.004981,0.000234,388282.173589,19025.842729,0.005227,0.000253,383733.366363,...,0.032579,22261.863303,0.000298,0.004582,0.005749,340100.114289,427366.618437,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C
9,2021q2,394196.946396,14786.349696,0.005154,0.000190,417108.094612,15825.376178,0.005453,0.000204,410629.575872,...,0.032050,20260.620901,0.000263,0.004854,0.005883,370918.758907,450340.392837,MODEL_BASED_HISTORICAL_COMPATIBILITY,False,C


## 7. Gates finais e freeze

In [9]:
FINAL_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_FINAL_LOCK.json'
HARDENING_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_HARDENING_LOCK.json'

final_lock = json.loads(FINAL_LOCK.read_text(encoding='utf-8'))
hardening_lock = json.loads(HARDENING_LOCK.read_text(encoding='utf-8'))

print(json.dumps(final_lock, ensure_ascii=False, indent=2))
assert final_lock['status'] == 'FINAL_CERTIFIED'
assert final_lock['certification_grade'] == 'PUBLICATION'
assert int(final_lock['bootstrap_reps_successful']) >= 475
assert hardening_lock['critical_failures'] == []
assert hardening_lock['run_id'] == RUN_ID

mca_hash = hardening_lock['artifact_hashes']['mca_cells']
support_hash = hardening_lock['artifact_hashes']['support']
assert mca_hash != support_hash, 'MCA e support continuam materialmente duplicados.'

print('PUBLICATION BACKCAST CERTIFIED')

{
  "freeze_id": "publication_20260724T185511Z_v120",
  "status": "FINAL_CERTIFIED",
  "component": "PNADC_HISTORICAL_BACKCAST",
  "hardening_lock": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_HISTORICAL_BACKCAST_HARDENING_LOCK.json",
  "upstream_core_freeze": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_HISTORICAL_PROXY_CORE_FREEZE.json",
  "final_estimates": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_historical_backcast_hardening/pnadc_historical_backcast_final_estimates_publication_20260724T185511Z_v120.csv",
  "final_estimates_sha256": "63624a8149820481760c8a0907c6093683891f7699eeeb80ddda28db86dd570b",
  "model_bundle": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/models/pnadc_historical_backcast_hardening/pnadc_historical_backcast_hardening_bundle_publication_20260724T185511Z_v120.joblib",
  "model_bundle_sha256": "c008b9e144adcbb60b6ec1bfe49863cce6207750d4177b622fd557f36f66

## 8. Iniciar um novo rerun no futuro

Após concluir e arquivar este run, remova ou renomeie `00_admin/PNADC_BACKCAST_PUBLICATION_RUN_ID_v120.txt` para gerar um novo `RUN_ID`.